In [2]:
import pandas as pd

df = pd.read_csv("../datasets/depression_dataset.csv")

In [3]:
print(df.head())

                                          clean_text  is_depression
0  we understand that most people who reply immed...              1
1  welcome to r depression s check in post a plac...              1
2  anyone else instead of sleeping more when depr...              1
3  i ve kind of stuffed around a lot in my life d...              1
4  sleep is my greatest and most comforting escap...              1


In [4]:
print(df.columns)

Index(['clean_text', 'is_depression'], dtype='str')


In [5]:
print(df.shape)

(7731, 2)


In [6]:
print(df.isnull().sum())

clean_text       0
is_depression    0
dtype: int64


In [7]:
df["is_depression"].value_counts()

is_depression
0    3900
1    3831
Name: count, dtype: int64

In [8]:
import pandas as pd

from sklearn.model_selection import train_test_split

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

import torch

c:\abhishek_projects\DepressionAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
df = pd.read_csv(
    "../datasets/depression_dataset.csv"
)

print(df.shape)

df.head()

(7731, 2)


,clean_text,is_depression
0,we understand that most people who reply immed...,1
1,welcome to r depression s check in post a plac...,1
2,anyone else instead of sleeping more when depr...,1
3,i ve kind of stuffed around a lot in my life d...,1
4,sleep is my greatest and most comforting escap...,1


In [10]:
X = df["clean_text"]

y = df["is_depression"]

In [11]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [13]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True
)

In [14]:
class DepressionDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings

        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels.iloc[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [15]:
train_dataset = DepressionDataset(
    train_encodings,
    train_labels
)

test_dataset = DepressionDataset(
    test_encodings,
    test_labels
)

In [16]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4218.65it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

In [18]:
training_args = TrainingArguments(

    output_dir="../outputs/depression_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    num_train_epochs=3,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    weight_decay=0.01,

    logging_dir="../outputs/logs",

    load_best_model_at_end=True
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [19]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [22]:
trainer.train()

c:\abhishek_projects\DepressionAI\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.136516,0.062858,0.977376,0.976867
2,0.041303,0.093092,0.985133,0.984858
3,0.005446,0.091917,0.981900,0.981747


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.05it/s]
c:\abhishek_projects\DepressionAI\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.64it/s]
c:\abhishek_projects\DepressionAI\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.14it/s]


TrainOutput(global_step=2319, training_loss=0.062200380636217675, metrics={'train_runtime': 15922.6675, 'train_samples_per_second': 1.165, 'train_steps_per_second': 0.146, 'total_flos': 2467072832569344.0, 'train_loss': 0.062200380636217675, 'epoch': 3.0})

In [ ]:
model.save_pretrained(
    "../models/depression_model"
)

tokenizer.save_pretrained(
    "../models/depression_model"
)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.21it/s]


('../models/depression_model\\tokenizer_config.json',
 '../models/depression_model\\tokenizer.json')

In [ ]:
trainer.evaluate()

c:\abhishek_projects\DepressionAI\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.007708,0.067958,3,0.978022,0.977602


{'eval_loss': 0.06795752048492432,
 'eval_accuracy': 0.978021978021978,
 'eval_f1': 0.9776021080368906}

In [21]:
print(df[df["is_depression"] == 1].head())

print(df[df["is_depression"] == 0].head())

                                          clean_text  is_depression
0  we understand that most people who reply immed...              1
1  welcome to r depression s check in post a plac...              1
2  anyone else instead of sleeping more when depr...              1
3  i ve kind of stuffed around a lot in my life d...              1
4  sleep is my greatest and most comforting escap...              1
                                             clean_text  is_depression
3831  switchfoot http twitpic com y zl awww that s a...              0
3832  is upset that he can t update his facebook by ...              0
3833  kenichan i dived many time for the ball manage...              0
3834       my whole body feel itchy and like it on fire              0
3835  nationwideclass no it s not behaving at all i ...              0
